# BL importer finalization & debug

## Imports

In [42]:
import os
import json

from impresso_essentials.utils import ALL_MEDIA, PARTNER_TO_MEDIA
from copy import deepcopy
from tqdm import tqdm
import pandas as pd
from random import shuffle
from text_preparation.importers.bl.detect import BlIssueDir, dir2issue, detect_issues, select_issues
from text_preparation.importers.bl.omni.classes import BlOmniNewspaperPage, BlOmniNewspaperIssue
from PIL import Image, ImageDraw, ImageFont
from text_preparation.utils import draw_box_on_img, coords_to_xywh, coords_to_xy, rescale_coords
from text_preparation.importers import (
    CONTENTITEM_TYPES,
    CONTENTITEM_TYPE_IMAGE,
    CONTENTITEM_TYPE_ADVERTISEMENT,
)

## OmniPage Format

First, adapt BL_ocr_formats.json file to only keep the Aliases and issues corresponding to OmniPage-NLP format.
That will allow to only detect titles for this format and will already help a lot.
The document could have all the issues or be much much smaller with only Alias > NLP > list of dates for the given format. Or directly have the list of paths to mets files of the correct format.

Then the BL_extended_title_list.csv should also be processed to go from alias > NLP > date range > working title and variant titles so that each issue can have its variant title attached to it.

In [2]:
bl_source_data_dir = "/mnt/project_impresso/original/BL"
bl_w_source_data_dir = "/mnt/impresso_ocr_BL"
bl_sample_dir = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL"
bl_formats_filename = "BL_ocr_formats.json"
bl_titles_filename = "BL_extended_title_list.csv"
alias_to_NLP_filename = "BL_alias_to_NLP.json"
bl_titles_out_filename = "BL_all_titles.json"
bl_alias_25_08_formats = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL/BL_ocr_formats_bl_alias_2025-08-25.json"
bl_media_list_ext_path = os.path.join(bl_sample_dir, bl_titles_filename)
alias_to_NLP = os.path.join(bl_sample_dir, alias_to_NLP_filename)
bl_format_specific_issues = "BL_{format}_issues.json"
RENAMING_INFO_FILE = "renaming_info.json"
BL_IMG_TYPE = "illustration"
BL_AD_TYPE = "advert"
BL_CAPTION_TYPE = "caption"

ocr_formats = ["OmniPage-NLP", "BL-ALIAS", "Nuance-NLP", "ABBYY-ALIAS", "ABBYY-NLP"]

In [3]:
with open(os.path.join(bl_source_data_dir, bl_formats_filename), "r", encoding='utf-8') as fin:
    bl_ocr_formats = json.load(fin)

print(f"Reading: {os.path.join(bl_source_data_dir, bl_formats_filename)}")

print(f"There are {len(bl_ocr_formats)} aliases in {bl_formats_filename}")

Reading: /mnt/project_impresso/original/BL/BL_ocr_formats.json
There are 368 aliases in BL_ocr_formats.json


##### In the original run of ocr formats identification, there was a small bug for Alias `HPNW`, (BL-alias, but listed as UNKNOWN in the original `bl_ocr_formats.json` file) so it needs to be corrected

In [ ]:
#with open(os.path.join(bl_source_data_dir, bl_formats_filename), "r", encoding='utf-8') as fin:
with open(os.path.join(bl_sample_dir, "BL_ocr_formats_old.json"), "r", encoding='utf-8') as fin:
    bl_ocr_formats = json.load(fin)

print(f"Reading: {os.path.join(bl_source_data_dir, bl_formats_filename)}")

# performing a deep copy of the original dict, in order to 
corrected_bl_ocr_formats = deepcopy(bl_ocr_formats)

print(f"There are {len(bl_ocr_formats)} aliases in {bl_formats_filename}")

In [ ]:
def id_bl_alias_format(alias, ocr_files, ocr_issue_dir, ocr_formats) -> dict[str, list]:
    _, nlp, year, month, day = ocr_issue_dir.split(alias)[-1].split("/")
    if alias == "HPNW":
        # there is a mismatch for this alias
        expected_prefix = f"WO1_IPNW_{year}_{month}_{day}-"
    else:
        expected_prefix = f"WO1_{alias}_{year}_{month}_{day}-"
    matching_files = [f for f in ocr_files if expected_prefix in f and f.endswith(".xml")]

    if matching_files:
        # add the matching files to the BL-ALIAS OCR format if there are any
        ocr_formats["BL-ALIAS"] = matching_files

    return ocr_formats

Re-identify the format for this alias, and ensure no issues/files are left unidentified

In [ ]:
incomplete_issues = []
other_formats_issues = []

for year, issues in bl_ocr_formats['HPNW'].items():
    for issue, formats in issues.items():
        if 'UNKNOWN' in formats:
            new_ocr_formats = id_bl_alias_format('HPNW', formats['UNKNOWN'], issue, {})
            if any(x not in new_ocr_formats["BL-ALIAS"] for x in formats['UNKNOWN']):
                print(f"NOTE - {year}-{issue} - More formats to indentify in addition to BL_ALIAS!!")
                incomplete_issues.append(issue)
            corrected_bl_ocr_formats['HPNW'][year][issue] = new_ocr_formats
        if len(formats) != 1:
            # ensure that any other formats are kept
            print(f"WARNING - {year}-{issue} - other formats than unknown found: {formats}")
            other_formats_issues.append(issue)
            for format, files in formats.items():
                if format != 'UNKNOWN':
                    corrected_bl_ocr_formats['HPNW'][year][issue][format]=files
                

incomplete_issues, other_formats_issues

WARNING - 1873-/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/08 - other formats than unknown found: {'ABBYY-NLP': ['0000072_18730208_mets.xml'], 'UNKNOWN': ['WO1_IPNW_1873_02_08-0002-006.xml', 'WO1_IPNW_1873_02_08-0004-025.xml', 'WO1_IPNW_1873_02_08-0004-026.xml', 'WO1_IPNW_1873_02_08-0004-023.xml', 'WO1_IPNW_1873_02_08-0002.xml', 'WO1_IPNW_1873_02_08-0002-011.xml', 'WO1_IPNW_1873_02_08-0002-008.xml', 'WO1_IPNW_1873_02_08-0001-004.xml', 'WO1_IPNW_1873_02_08-0001.xml', 'WO1_IPNW_1873_02_08-0004-024.xml', 'WO1_IPNW_1873_02_08-0004-020.xml', 'WO1_IPNW_1873_02_08-0004-021.xml', 'WO1_IPNW_1873_02_08-0003-019.xml', 'WO1_IPNW_1873_02_08-0002-007.xml', 'WO1_IPNW_1873_02_08-0002-014.xml', 'WO1_IPNW_1873_02_08-0001-002.xml', 'WO1_IPNW_1873_02_08-0004.xml', 'WO1_IPNW_1873_02_08-0003-018.xml', 'WO1_IPNW_1873_02_08-0002-010.xml', 'WO1_IPNW_1873_02_08-0004-027.xml', 'WO1_IPNW_1873_02_08-0002-013.xml', 'WO1_IPNW_1873_02_08-0003-015.xml', 'WO1_IPNW_1873_02_08-0003-017.xml', 'WO1_IPNW_1873_02_

([],
 ['/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/08',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/15',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/01',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/02/22',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/17',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/24',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/03',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/10',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/05/31',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/02',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/23',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/30',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/09',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/08/16',
  '/mnt/project_impresso/original/BL/HPNW/0000072/1873/11/08',
  '/mnt/project_impresso/original/BL/HPNW/0000072/

In [37]:
len(other_formats_issues)

51

In [ ]:
corrected_bl_ocr_formats['HPNW']['1873']['/mnt/project_impresso/original/BL/HPNW/0000072/1873/11/08']

Ensure that the new corrected version of `BL_ocr_formats` has exactly the same keys and values for all aliases, except the alias in question.

In [33]:
for alias, formats in bl_ocr_formats.items():
    if corrected_bl_ocr_formats[alias] != formats:
        same_keys = corrected_bl_ocr_formats[alias].keys() == formats.keys()
        same_issues = all(issues.keys()==formats[year].keys() for year, issues in corrected_bl_ocr_formats[alias].items())
        print(f"Different formats for alias: {alias}, same keys: {same_keys}, same issues: {same_issues}")

len(bl_ocr_formats), len(corrected_bl_ocr_formats)

Different formats for alias: HPNW, same keys: True, same issues: True


(368, 368)

Now the new version of bl_ocr_formats can be saved in the two places it exists

In [34]:
print(f"writing the corrected OCR formats file to paths {os.path.join(bl_w_source_data_dir, bl_formats_filename)} and {os.path.join(bl_sample_dir, bl_formats_filename)}")

writing the corrected OCR formats file to paths /mnt/impresso_ocr_BL/BL_ocr_formats.json and /home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/BL/BL_ocr_formats.json


In [35]:
with open(os.path.join(bl_w_source_data_dir, bl_formats_filename), "w", encoding='utf-8') as fout:
    json.dump(corrected_bl_ocr_formats, fout, indent=2)

with open(os.path.join(bl_sample_dir, bl_formats_filename), "w", encoding='utf-8') as fout:
    json.dump(corrected_bl_ocr_formats, fout, indent=2)

In [ ]:
len(bl_ocr_formats['HPNW']), bl_ocr_formats['HPNW']

### 1. Create format-specific json files

In [38]:
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    print(filepath)

/mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues.json
/mnt/project_impresso/original/BL/BL_BL-ALIAS_issues.json
/mnt/project_impresso/original/BL/BL_Nuance-NLP_issues.json
/mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues.json
/mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues.json


In [ ]:
mets_paths_per_format = {f: {} for f in ocr_formats}

for alias_idx, (alias, yearly_formats) in enumerate(corrected_bl_ocr_formats.items(), start=1):
    num_years = len(yearly_formats)
    print(f"Starting to process alias {alias} ({alias_idx}/368) - {num_years} years:")

    for year, issue_formats in tqdm(yearly_formats.items()):

        for issue_dir_path, formats in issue_formats.items():

            # keep track of which format is the first match for the priority list
            first_match = True
            for format in ocr_formats:

                if format in formats:
                    if alias not in mets_paths_per_format[format]:
                        # initialize the format dict for this alias is not already done
                        mets_paths_per_format[format][alias] = {
                            "priority_issues": {},
                            "other_issues_also_in_format": {},
                            "non_public_domain_post_1905": {}
                        }
                        
                    # select the mets filename or the first file (for BL alias case)
                    mets_or_example_file = formats[format][0]

                    # directly exclude any title after 1905
                    if int(year) > 1905:
                        mets_paths_per_format[format][alias]["non_public_domain_post_1905"][issue_dir_path] = mets_or_example_file
                    elif first_match:  
                        # set pairs of (issue_path, mets or example file)
                        mets_paths_per_format[format][alias]["priority_issues"][issue_dir_path] = mets_or_example_file
                        # all other formats for this issue are less of a priority
                        first_match = False
                    else:
                        mets_paths_per_format[format][alias]["other_issues_also_in_format"][issue_dir_path] = mets_or_example_file

    print(f"Saving to files the issues: latest alias {alias} ({alias_idx}/368).")

    for format in ocr_formats:
        filepath = os.path.join(bl_w_source_data_dir, bl_format_specific_issues.format(format=format))

        with open(filepath, "w", encoding="utf-8") as fout:
            json.dump(mets_paths_per_format[format], fout, indent=2)

Correcting accordingly the issue lists

In [42]:
bl_format_specific_issues_old = "BL_{format}_issues_old.json"
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    old_filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues_old.format(format=format))
    print(filepath, old_filepath)

/mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues.json /mnt/project_impresso/original/BL/BL_OmniPage-NLP_issues_old.json
/mnt/project_impresso/original/BL/BL_BL-ALIAS_issues.json /mnt/project_impresso/original/BL/BL_BL-ALIAS_issues_old.json
/mnt/project_impresso/original/BL/BL_Nuance-NLP_issues.json /mnt/project_impresso/original/BL/BL_Nuance-NLP_issues_old.json
/mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues.json /mnt/project_impresso/original/BL/BL_ABBYY-ALIAS_issues_old.json
/mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues.json /mnt/project_impresso/original/BL/BL_ABBYY-NLP_issues_old.json


In [43]:
old_issue_lists = {}
new_issue_lists = {}
for format in ocr_formats:
    filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues.format(format=format))
    old_filepath = os.path.join(bl_source_data_dir, bl_format_specific_issues_old.format(format=format))

    with open(filepath, "r", encoding="utf-8") as fin:
        new_issue_lists[format] = json.load(fin)
    with open(old_filepath, "r", encoding="utf-8") as fin:
        old_issue_lists[format] = json.load(fin)

    print(f"{format}: new_issue_lists==old_issue_lists: {new_issue_lists[format]==old_issue_lists[format]}")

OmniPage-NLP: new_issue_lists==old_issue_lists: True
BL-ALIAS: new_issue_lists==old_issue_lists: False
Nuance-NLP: new_issue_lists==old_issue_lists: True
ABBYY-ALIAS: new_issue_lists==old_issue_lists: True
ABBYY-NLP: new_issue_lists==old_issue_lists: False


### 2. Creating a file with all the variant titles

In [ ]:
bl_media_lst_ext_raw_df = pd.read_csv(bl_media_list_ext_path, header=0, index_col=0)
print(bl_media_lst_ext_raw_df.info())
bl_media_lst_ext_raw_df.head()

In [ ]:
cols_to_keep = ['Alias (in file-syst or generated)','Normalized Working Title', 'Working title (BL)', 'Variant Title', 'NLP', 'Time Period']

bl_media_lst_ext_raw_df['Time Period'] = bl_media_lst_ext_raw_df[['Start year in Impresso local copy', 'End year in Impresso local copy']].apply(lambda x: f"{x[0]}-{x[1]}", axis=1)
reduced_title_df = bl_media_lst_ext_raw_df[cols_to_keep]
reduced_title_df['NLP'] = reduced_title_df['NLP'].apply(lambda x: str(x).zfill(7))
reduced_title_df['Alias-NLP'] = reduced_title_df[['Alias (in file-syst or generated)', 'NLP']].apply(lambda x: f"{x[0]}-{x[1]}", axis=1)

for col in ['Normalized Working Title', 'Working title (BL)', "Variant Title"]:
    reduced_title_df[col] = reduced_title_df[col].apply(
        lambda x: x.rstrip('. ') if isinstance(x, str) else [y.rstrip('. ') for y in x]
    )

reduced_title_df

In [57]:
bl_titles = reduced_title_df.groupby(['Alias-NLP', 'Time Period']).agg({
        'Normalized Working Title': lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
        'Working title (BL)': lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
        "Variant Title": lambda x: x.unique()[0] if len(x.unique())==1 else x.unique(),
    },
).reset_index()

bl_titles

,Alias-NLP,Time Period,Normalized Working Title,Working title (BL),Variant Title
0,AATA-0003031,1846-1846,The Agricultural Advertiser and Tenant-Farmers...,Agricultural Advertiser and Tenant-Farmers' Ad...,The Agricultural Advertiser and Tenant-Farmers...
1,AGE52-0003023,1852-1853,The Age 1852,Age 1852,The Age
2,AGMO-0002364,1811-1811,The Anti-Gallican Monitor,Anti-Gallican Monitor,The Anti-Gallican Monitor
3,AGMO-0002365,1811-1814,The Anti-Gallican Monitor,Anti-Gallican Monitor,"The Anti-Gallican Monitor, and Anti-Corsican C..."
4,AGMO-0002366,1815-1817,The Anti-Gallican Monitor,Anti-Gallican Monitor,The Anti-Gallican Monitor
...,...,...,...,...,...
638,YOHD-0000497,1812-1813,The York Herald,York Herald,"The York Herald, County and General Advertiser"
639,YOHD-0000498,1814-1854,The York Herald,York Herald,The York Herald and General Advertiser
640,YOHD-0000499,1855-1889,The York Herald,York Herald,The York Herald
641,YOHD-0000500,1890-1900,The York Herald,York Herald,The Yorkshire Herald and the York Herald


In [ ]:
titles_as_dict = bl_titles.to_dict(orient='records')
titles_as_dict

In [63]:
bl_titles_json = {}
for record in titles_as_dict:
    if (
        not isinstance(record['Normalized Working Title'], str) or 
        not isinstance(record['Working title (BL)'], str) or 
        not isinstance(record['Variant Title'], str)
    ):
        print(f"More than one value! {record}")
        if record['Alias-NLP']=='BRLU-0002378' and record['Time Period']=='1820-1821':
            record['Variant Title']=record['Variant Title'][0]
        if (
            (record['Alias-NLP']=='DCWR-0003408' and record['Time Period']=='1869-1895') or 
            (record['Alias-NLP']=='MEXA-0003398' and record['Time Period']=='1846-1848')
        ):
            record['Normalized Working Title']=record['Normalized Working Title'][1]
            record['Variant Title']=record['Variant Title'][1]
            record['Working title (BL)']=record['Working title (BL)'][1]
    bl_titles_json.update({
       record['Alias-NLP'] : {record['Time Period']: record}
    })

with open(os.path.join(bl_w_source_data_dir, bl_titles_out_filename), "w", encoding='utf-8') as fout:
    json.dump(bl_titles_json, fout, indent=2)

More than one value! {'Alias-NLP': 'BRLU-0002378', 'Time Period': '1820-1821', 'Normalized Working Title': 'The British Luminary', 'Working title (BL)': 'British Luminary', 'Variant Title': array(['The Weekly Intelligencer, and British Luminary',
       'The Weekly Intelligencer, and British Luminary. (30 July 1820-27 May 1821)'],
      dtype=object)}
More than one value! {'Alias-NLP': 'DCWR-0003408', 'Time Period': '1869-1895', 'Normalized Working Title': array(['Dewsbury Chronicle and West Riding Advertiser',
       'The Dewsbury Chronicle and West Riding Advertiser'], dtype=object), 'Working title (BL)': array(['Dewsbury Chronicle and West Riding Advertiser',
       'The Dewsbury Chronicle and West Riding Advertiser'], dtype=object), 'Variant Title': array(['Dewsbury Chronicle and West Riding Advertiser',
       'The Dewsbury Chronicle, and West Riding Advertiser'], dtype=object)}
More than one value! {'Alias-NLP': 'MEXA-0003398', 'Time Period': '1846-1848', 'Normalized Working Titl

### 3. Detect/Select functions

In [ ]:
omni_issues = mets_paths_per_format["OmniPage-NLP"]
omni_issues

In [ ]:
all_issues = []
for alias, issues_of_alias in omni_issues.items():
    
    issue_paths = [dir2issue(path) for path in list(issues_of_alias['priority_issues'].keys())]
    print(f"{alias} - Found {len(issue_paths)} issues")
    all_issues.extend(issue_paths)

all_issues[10:-10]

In [4]:
config_test_1 = {
    "titles": {
        "ILOL": [],
        "AGMO": [],
        "AATA": [],
        "AGE52": [],
        "AHEC": [],
        "ANJO": [],
        "BHCH": [],
    },
    "exclude_titles": [],
    "year_only": False
}

detected = detect_issues(bl_source_data_dir)

print(f"Detected {len(detected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in detected if i.alias == title])} issues for {title}")

Detected 161486 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [5]:
selected = select_issues(bl_source_data_dir, config = config_test_1)

print(f"Selected  {len(selected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Selected  990 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [16]:
config_test_2 = {
    "titles": {
        "AGMO": "1817/08/03-1817/12/21",
        "AATA": "1848/01/01-1849/01/01",
        "AGE52": [],
        "AHEC": ["1880/02/21", "1877/02/10"],
        "ANJO": [],
        "BHCH": [],
    },
    "exclude_titles": [],
    "year_only": False
}

selected = select_issues(bl_source_data_dir, config = config_test_2)

print(f"Selected  {len(selected )} issues in total")

for title in config_test_2['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Selected  60 issues in total
Detected 21 issues for AGMO
Detected 0 issues for AATA
Detected 37 issues for AGE52
Detected 2 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


### 4. BlOmniNewspaperIssue Class

In [4]:
titles_for_alias_nlp = {
    "1801-1811": {
      "Alias-NLP": "YOHD-0000186",
      "Time Period": "1801-1811",
      "Normalized Working Title": "The York Herald",
      "Working title (BL)": "York Herald",
      "Variant Title": "The York Herald"
    }
  }

period = [int(y) for p in titles_for_alias_nlp for y in p.split("-")]
period

[1801, 1811]

#### Debug issue

In [4]:
config_test_1 = {
    "titles": {
        "ILOL": [],
        "AGMO": [],
        "AATA": [],
        "AGE52": [],
        "AHEC": [],
        "ANJO": [],
        "BHCH": [],
    },
    "exclude_titles": [],
    "year_only": False
}

detected = detect_issues(bl_source_data_dir)

print(f"Detected {len(detected)} issues in total")

selected = select_issues(bl_source_data_dir, config = config_test_1)

print(f"Selected  {len(selected)} issues in total")

for title in config_test_1['titles']:
    print(f"Detected {len([i for i in selected if i.alias == title])} issues for {title}")

Detected 161486 issues in total
Selected  990 issues in total
Detected 23 issues for ILOL
Detected 691 issues for AGMO
Detected 31 issues for AATA
Detected 37 issues for AGE52
Detected 208 issues for AHEC
Detected 0 issues for ANJO
Detected 0 issues for BHCH


In [5]:
test_issue = selected[0]
test_issue

IssueDirectory(provider='BL', alias='ILOL', date=datetime.date(1843, 5, 21), edition='a', path='/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21', nlp='0003005')

In [6]:
os.listdir(test_issue.path)

['renaming_info.json',
 '0003005_18430521_0007.xml',
 '0003005_18430521_0009.xml',
 '0003005_18430521_0011.xml',
 '0003005_18430521_0008.xml',
 '0003005_18430521_0006.xml',
 '0003005_18430521_0012.xml',
 '0003005_18430521_0004.xml',
 '0003005_18430521_mets.xml',
 '0003005_18430521_0003.xml',
 '0003005_18430521_0002.xml',
 '0003005_18430521_0005.xml',
 '0003005_18430521_0010.xml',
 '0003005_18430521_0001.xml']

In [7]:
with open(os.path.join(test_issue.path, RENAMING_INFO_FILE), 'r') as fin:
    test_issue_renaming_info = json.load(fin)

test_issue_renaming_info

{'12': {'original_filename': '0003005_18430521_0012.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0012.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '1': {'original_filename': '0003005_18430521_0001.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0001.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '5': {'original_filename': '0003005_18430521_0005.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0005.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843

In [8]:
bl_issue = BlOmniNewspaperIssue(test_issue)
bl_issue

In [9]:
bl_issue.issue_data

{'id': 'ILOL-1843-05-21-a',
 'cdt': '2025-08-27 18:07:52',
 'ts': '2025-08-27T16:07:52Z',
 'st': 'newspaper',
 'sm': 'print',
 'olr': True,
 'i': [{'m': {'id': 'ILOL-1843-05-21-a-i0001',
    'tp': 'article',
    'pp': [1],
    'var_t': 'Illustrated London Life',
    'lg': 'en',
    'ro': 1},
   'l': {'bl_nlp': '0003005',
    'src_files': {'mets_xml': '0003005_18430521_mets.xml',
     'alto_xml': ['0003005_18430521_0001.xml'],
     'page_image': ['0003005_18430521_0001.jp2']},
    'id': 'art0001',
    'parts': [{'comp_role': 'pagearea',
      'comp_id': 'pa0001001',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001002',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role': 'pagearea',
      'comp_id': 'pa0001003',
      'comp_label': 'textblock',
      'comp_fileid': 'img0001-alto',
      'comp_page_no': 1},
     {'comp_role':

##### Adapt `_parse_content_item_logic()` for handle images and store the original filename

In [ ]:
mets_doc = bl_issue.xml
content_items = []

# Get logical structure of issue
divs = (
    mets_doc.find("structMap", {"TYPE": "LOGICAL"})
    .find("div", {"TYPE": "ISSUE"})
    .findChildren("div")
)

# Sort to have same naming
#sorted_divs = sorted(divs, key=lambda x: int(''.join(filter(str.isdigit, x.get("DMDID")))))

#sorted_divs
divs

In [13]:
def all_equal(iterator):
    iterator = iter(iterator)
    try:
        first = next(iterator)
    except StopIteration:
        return True
    return all(first == x for x in iterator)

In [ ]:
found_types = set(x.get("TYPE") for x in divs)

phys_structmap = mets_doc.find("structMap", {"TYPE": "PHYSICAL"})
structlink = mets_doc.find("structLink")

print(found_types)
print(phys_structmap)
print(structlink)

##### `_parse_content_item`

In [15]:
def _get_part_dict(div, comp_role: str | None):
    """Construct the parts for a certain div entry of METS.

    Args:
        div (Tag): Content item div
        comp_role (str | None): Role of the component

    Returns:
        dict[str, Any]: Parts dict for given div.
    """
    comp_fileid = div.find("area", {"BETYPE": "IDREF"}).get("FILEID")
    comp_id = div.get("ID")
    comp_page_no = int(div.parent.get("ORDER"))
    # This is where illustrations will be identified
    comp_label = div.get("LABEL").lower()
    if comp_role is None:
        type_attr = div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

    return {
        "comp_role": comp_role,
        "comp_id": comp_id,
        "comp_label": comp_label,
        "comp_fileid": comp_fileid,
        "comp_page_no": int(comp_page_no),
    }

In [38]:
def _get_image_and_captions(div, part_id, div_parts, curr_ci_parts, ci_image_parts, last_img_part_id):
    # for each illustration, store its coordinates and any potential caption
    if div.get("LABEL").lower() == BL_IMG_TYPE:
        img_xy_coords = div.find("area", {"SHAPE": "RECT"}).get("COORDS")
        # directly convert the coordinates to the wanted xywh format
        div_parts["coords"] = coords_to_xywh([int(c) for c in img_xy_coords.split(',')])
        if part_id not in ci_image_parts:
            ci_image_parts[part_id] = [div_parts]
        else:
            ci_image_parts[part_id].append(div_parts)
        # keep track of which illustration it is to make sure we can connect them back after
        last_img_part_id = part_id

    # if the next element is a caption, attach it directly
    if div.get("LABEL").lower() == BL_CAPTION_TYPE:
        if curr_ci_parts[-1]['comp_id'] == last_img_part_id:
            #ci_image_parts[last_img_part_id]["caption_parts"] = div_parts
            cap_xy_coords = div.find("area", {"SHAPE": "RECT"}).get("COORDS")
            # directly convert the coordinates to the wanted xywh format
            div_parts["coords"] = coords_to_xywh([int(c) for c in cap_xy_coords.split(',')])
            ci_image_parts[last_img_part_id].append(div_parts)
            #ci_image_parts[last_img_part_id]["caption_coords"] = coords_to_xywh([int(c) for c in cap_xy_coords.split(',')])
        else:
            print(f"curr_ci_parts[-1]: {curr_ci_parts[-1]}, last_img_part_id={last_img_part_id}")
            msg = f"{bl_issue.id}, {div_parts['comp_page_no']} - caption {div.get('ID')} does not follow an illustration!"
            print(msg)
            bl_issue._notes.append(msg)

    return ci_image_parts, last_img_part_id

In [27]:
counter = 1

item_div = divs[0]
#for div in divs:
#only for first for now
    # Parse Each contentitem
item_dmd_sec = mets_doc.find("dmdSec", {"ID": item_div.get("DMDID")})
    #content_items.append(test_issue._parse_content_item(div, counter, phys_structmap, structlink, dmd_sec))
 
div_type = item_div.get("TYPE").lower()

div_id = item_div.get("ID")

lang = item_dmd_sec.findChild("languageTerm").text

div_id, item_dmd_sec, div_type, lang

('art0001',
 <mets:dmdSec ID="modsarticle1">
 <mets:mdWrap MDTYPE="MODS">
 <mets:xmlData>
 <mods:mods>
 <mods:language>
 <mods:languageTerm authority="rfc3066" type="code">en</mods:languageTerm>
 </mods:language>
 </mods:mods>
 </mets:xmlData>
 </mets:mdWrap>
 </mets:dmdSec>,
 'article',
 'en')

In [ ]:
counter = 1
content_items = []
cis_img_parts = []

for idx, div in enumerate(divs):
    print(f"\n---- div #{idx} -----")
    print(f"counter = {counter}")
    dmd_sec = mets_doc.find("dmdSec", {"ID": div.get("DMDID")})
    div_type = div.get("TYPE").lower()

    item_dmd_sec = mets_doc.find("dmdSec", {"ID": div.get("DMDID")})
    lang = item_dmd_sec.findChild("languageTerm")

    if div_type == BL_IMG_TYPE:
        div_type = CONTENTITEM_TYPE_IMAGE
    elif div_type == BL_AD_TYPE:
        div_type = CONTENTITEM_TYPE_ADVERTISEMENT
    
    tag = tag = f"#{div.get('ID')}"
    print(tag)
    linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent

    # Remove `#` from xlink:href
    div_parts_ids = [
        x.get("xlink:href")[1:]
        for x in linkgrp.findAll("smLocatorLink")
        if x.get("xlink:href") != tag
    ]

    ci_parts = []
    ci_image_parts = {}
    last_img_part_id = None
    last_img_part_idx = None
    for idx, p in enumerate(div_parts_ids):
        # Get element in physical map
        part_div = phys_structmap.find("div", {"ID": p})
        print(f"div {idx} from div parts: {div}")
        type_attr = part_div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

        # In that case, need to add all parts
        if comp_role == "page":
            for sub_div in part_div.findAll("div"):
                subdiv_part_dict = _get_part_dict(sub_div, None)
                subdiv_part_id = sub_div.get("ID")
                assert subdiv_part_id == subdiv_part_dict['comp_id']#:
                #print(f"!!!!!!!subdiv_part_id={subdiv_part_id}, subdiv_part_dict['comp_id']:{subdiv_part_dict['comp_id']}")
                
                ci_image_parts, last_img_part_id = _get_image_and_captions(sub_div, subdiv_part_id, subdiv_part_dict, ci_parts, ci_image_parts, last_img_part_id)
                ci_parts.append(subdiv_part_dict)
                
        else:
            div_part_dict = _get_part_dict(part_div, comp_role)
            ci_image_parts, last_img_part_id = _get_image_and_captions(part_div, p, div_part_dict, ci_parts, ci_image_parts, last_img_part_id)
            ci_parts.append(div_part_dict)

    
    content_item = {
        "m": {
            "id": f"{bl_issue.id}-i{str(counter).zfill(4)}",
            "tp": div_type,
            "pp": [],
        },
        "l": {
            "bl_nlp": bl_issue.nlp,
            "id": div.get("ID"),
            "parts": ci_parts,
        },
    }

    if lang is not None:
        content_item['m']["lg"] = lang.text
    for p in content_item["l"]["parts"]:
        pge_no = p["comp_page_no"]
        if pge_no not in content_item["m"]["pp"]:
            content_item["m"]["pp"].append(pge_no)


    content_items.append(content_item)
    cis_img_parts.append(ci_image_parts)
    counter += 1

In [40]:
content_items[43]

{'m': {'id': 'ILOL-1843-05-21-a-i0044',
  'tp': 'article',
  'pp': [11],
  'lg': 'en'},
 'l': {'bl_nlp': '0003005',
  'id': 'art0044',
  'parts': [{'comp_role': 'pagearea',
    'comp_id': 'pa0011003',
    'comp_label': 'headline',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011004',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011005',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011006',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011007',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-alto',
    'comp_page_no': 11},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0011008',
    'comp_label': 'textblock',
    'comp_fileid': 'img0011-a

In [41]:
cis_img_parts[43]#38

{'pa0011023': [{'comp_role': 'pagearea',
   'comp_id': 'pa0011023',
   'comp_label': 'illustration',
   'comp_fileid': 'img0011-alto',
   'comp_page_no': 11,
   'coords': [2887, 2183, 789, 575]},
  {'comp_role': 'pagearea',
   'comp_id': 'pa0011024',
   'comp_label': 'caption',
   'comp_fileid': 'img0011-alto',
   'comp_page_no': 11,
   'coords': [3047, 2805, 525, 38]}]}

In [26]:
bl_issue.pages[9]

In [21]:
issue_images_path = "/mnt/impresso_images_BL/ILOL/1843/05/21/a"
pg_filename = 'ILOL-1843-05-21-a-p{num}.jp2'
pg_path = os.path.join(issue_images_path, pg_filename)

In [ ]:
ci_idx = 0
ci = content_items[ci_idx]
img_part = cis_img_parts[ci_idx]
page_n = ci['m']['pp'][0]
pg_1_path = pg_path.format(num=str(page_n).zfill(4))

pg_1_img = Image.open(pg_1_path)
for div_id, parts in img_part.items():
    coords_img_xy = coords_to_xy(parts['coords'])
    print(f"coords_img_xy:{coords_img_xy}")
    pg_1_img = draw_box_on_img(pg_1_path, coords_img_xy, pg_1_img, width=15)
    if 'caption_coords' in parts:
        coords_cap_xy = coords_to_xy(parts['caption_coords'])
        print(f"coords_cap_xy:{coords_cap_xy}")
        #coords_cap_xy = coords_to_xy([int(c) for c in str_coords])
        pg_1_img = draw_box_on_img(pg_1_path, coords_cap_xy, pg_1_img, width=10)
    print(f"showing with {div_id}")
    pg_1_img.show()

In [ ]:
ci_idx = 43
ci = content_items[ci_idx]
img_part = cis_img_parts[ci_idx]
page_n = ci['m']['pp'][0]
pg_1_path = pg_path.format(num=str(page_n).zfill(4))

pg_1_img = Image.open(pg_1_path)
for div_id, parts in img_part.items():
    coords_img_xy = [int(c) for c in parts['coords'].split(',')]
    #coords_img_xy = coords_to_xy([int(c) for c in str_coords])
    coords_img_xywh = coords_to_xywh(coords_img_xy)
    print(f"coords_img_xy:{coords_img_xy}, coords_img_xywh:{coords_img_xywh}")
    pg_1_img = draw_box_on_img(pg_1_path, coords_img_xy, pg_1_img, width=15)
    if 'caption_coords' in parts:
        coords_cap_xy = [int(c) for c in parts['caption_coords'].split(',')]
        #coords_cap_xy = coords_to_xy([int(c) for c in str_coords])
        coords_cap_xywh = coords_to_xywh(coords_cap_xy)
        print(f"coords_cap_xy:{coords_cap_xy}, coords_cap_xywh:{coords_cap_xywh}")
        pg_1_img = draw_box_on_img(pg_1_path, coords_cap_xy, pg_1_img, width=10)
    print(f"showing with {div_id}")
    pg_1_img.show()

In [ ]:
problem_div = divs[37]
problem_div

tag = tag = f"#{problem_div.get('ID')}"
linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent
print(tag, linkgrp)

pb_div_parts_ids = [
    x.get("xlink:href")[1:]
    for x in linkgrp.findAll("smLocatorLink")
    if x.get("xlink:href") != tag
]

print(f"pb_div_parts = {pb_div_parts_ids}")

prb_div_part_div = phys_structmap.find("div", {"ID": pb_div_parts_ids[0]})

comp_role = prb_div_part_div.get('TYPE').lower() if type_attr else None

print(f"comp_role = {comp_role}")

prb_div_part_div

##### `_parse_content_parts`

In [ ]:
tag = tag = f"#{item_div.get('ID')}"
print(tag)

linkgrp = structlink.find("smLocatorLink", {"xlink:href": tag}).parent

# Remove `#` from xlink:href
parts_ids = [
    x.get("xlink:href")[1:]
    for x in linkgrp.findAll("smLocatorLink")
    if x.get("xlink:href") != tag
]

parts_ids

#art0001
<mets:smLinkGrp>
<mets:smLocatorLink xlink:href="#art0001" xlink:label="article" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001001" xlink:label="page1 area1" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001002" xlink:label="page1 area2" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001003" xlink:label="page1 area3" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001004" xlink:label="page1 area4" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001005" xlink:label="page1 area5" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001006" xlink:label="page1 area6" xlink:type="locator"/>
<mets:smLocatorLink xlink:href="#pa0001007" xlink:label="page1 area7" xlink:type="locator"/>
<mets:smArcLink ARCTYPE="logicalphysical" xlink:from="article" xlink:to="page1 area1" xlink:type="arc"/>
<mets:smArcLink ARCTYPE="logicalphysical" xlink:from="article" xlink:to="page1 area2" xlink:type="arc"/>
<mets:smArcLink ARCTYPE="l

['pa0001001',
 'pa0001002',
 'pa0001003',
 'pa0001004',
 'pa0001005',
 'pa0001006',
 'pa0001007']

In [15]:
def _get_part_dict(div, comp_role: str | None):
    """Construct the parts for a certain div entry of METS.

    Args:
        div (Tag): Content item div
        comp_role (str | None): Role of the component

    Returns:
        dict[str, Any]: Parts dict for given div.
    """
    comp_fileid = div.find("area", {"BETYPE": "IDREF"}).get("FILEID")
    comp_id = div.get("ID")
    comp_page_no = int(div.parent.get("ORDER"))
    # This is where illustrations will be identified
    comp_label = div.get("LABEL").lower()
    if comp_role is None:
        type_attr = div.get("TYPE")
        comp_role = type_attr.lower() if type_attr else None

    return {
        "comp_role": comp_role,
        "comp_id": comp_id,
        "comp_label": comp_label,
        "comp_fileid": comp_fileid,
        "comp_page_no": int(comp_page_no),
    }

In [17]:
div = phys_structmap.find("div", {"ID": parts_ids[0]})

div.get("LABEL")

'Textblock'

In [24]:
found_divs = phys_structmap.find_all('div', {"ID": lambda x: x in parts_ids})

all(d.get('LABEL') for d in found_divs)

True

In [16]:
parts = []
image_parts = {}
last_img_part_id = None
last_img_part_idx = None
for idx, p in enumerate(parts_ids):
    # Get element in physical map
    div = phys_structmap.find("div", {"ID": p})
    type_attr = div.get("TYPE")
    comp_role = type_attr.lower() if type_attr else None

    # In that case, need to add all parts
    if comp_role == "page":
        for x in div.findAll("div"):
            div_parts = _get_part_dict(x, None)
    else:
        div_parts = _get_part_dict(div, comp_role)
    
    # for each illustration, store its coordinates and any potential caption
    if div.get("LABEL").lower() == 'illustration':
        image_parts[p] = {
                "legacy_parts": div_parts,
                "coords": div.find("area", {"SHAPE": "RECT"}).get("COORDS"),
            }
        # keep track of which illustration it is to make sure we can connect them back after
        last_img_part_id = p
        last_img_part_idx = idx
    # if the next element is a caption, attach it directly
    if div.get("LABEL").lower() == 'caption':
        if idx-1 == last_img_part_idx:
            image_parts[last_img_part_id]['caption_parts'] = div_parts
        else:
            msg = f"self.id, {div_parts['comp_page_no']} - caption {div.get('ID')} does not follow an illustration!"
    
    parts.append(div_parts)

parts

[{'comp_role': 'pagearea',
  'comp_id': 'pa0001001',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001002',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001003',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001004',
  'comp_label': 'textblock',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001005',
  'comp_label': 'illustration',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001006',
  'comp_label': 'caption',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1},
 {'comp_role': 'pagearea',
  'comp_id': 'pa0001007',
  'comp_label': 'illustration',
  'comp_fileid': 'img0001-alto',
  'comp_page_no': 1}]

In [55]:
image_parts

{'pa0001005': {'legacy_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001005',
   'comp_label': 'illustration',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1},
  'coords': '298,4676,2539,5835',
  'caption_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001006',
   'comp_label': 'caption',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1}},
 'pa0001007': {'legacy_parts': {'comp_role': 'pagearea',
   'comp_id': 'pa0001007',
   'comp_label': 'illustration',
   'comp_fileid': 'img0001-alto',
   'comp_page_no': 1},
  'coords': '1649,4794,2147,4985'}}

In [53]:
content_item = {
    "m": {
        "id": f"{bl_issue.id}-i{str(counter).zfill(4)}",
        "tp": div_type,
        "pp": [],
    },
    "l": {
        "bl_nlp": bl_issue.nlp,
        "id": item_div.get("ID"),
        "parts": parts,
    },
}
for p in content_item["l"]["parts"]:
    pge_no = p["comp_page_no"]
    if pge_no not in content_item["m"]["pp"]:
        content_item["m"]["pp"].append(pge_no)

content_item

{'m': {'id': 'ILOL-1843-05-21-a-i0001', 'tp': 'article', 'pp': [1]},
 'l': {'bl_nlp': '0003005',
  'id': 'art0001',
  'parts': [{'comp_role': 'pagearea',
    'comp_id': 'pa0001001',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001002',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001003',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001004',
    'comp_label': 'textblock',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001005',
    'comp_label': 'illustration',
    'comp_fileid': 'img0001-alto',
    'comp_page_no': 1},
   {'comp_role': 'pagearea',
    'comp_id': 'pa0001006',
    'comp_label': 'caption',
    'comp_fileid': 'img0001-alto',
    'comp_page_n

In [ ]:
def _get_image_parts(parts_ids):
    """Construct the parts for a certain div entry of METS.

    Args:
        div (Tag): Content item div
        comp_role (str | None): Role of the component

    Returns:
        dict[str, Any]: Parts dict for given div.
    """
    comp_fileid = div.find("area", {"BETYPE": "IDREF"}).get("FILEID")
    comp_id = div.get("ID")
    comp_page_no = int(div.parent.get("ORDER"))
    # This is where illustrations will be identified
    comp_label = div.get("LABEL").lower()
    if comp_label == 'illustration':
        return div

In [ ]:
for p in
image_parts = [for p in parts_ids]

### 5. BlOmniNewspaperPage Class

#### Debug page

In [11]:
[page.number for page in bl_issue.pages]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

In [9]:
image_cis = [i for i in bl_issue.issue_data['i'] if i['m']['tp']=='image']
image_cis

[{'m': {'id': 'ILOL-1843-05-21-a-i0002',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-project.ch/api/proxy/iiif/ILOL-1843-05-21-a-p0001/info.json',
   'var_t': 'Illustrated London Life',
   'ro': 2},
  'l': {'bl_nlp': '0003005',
   'src_files': {'mets_xml': '0003005_18430521_mets.xml',
    'alto_xml': ['0003005_18430521_0001.xml'],
    'page_image': ['0003005_18430521_0001.jp2']},
   'id': 'pa0001005',
   'parts': [{'comp_role': 'pagearea',
     'comp_id': 'pa0001005',
     'comp_label': 'illustration',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1,
     'coords': [298, 4676, 2241, 1159]},
    {'comp_role': 'pagearea',
     'comp_id': 'pa0001006',
     'comp_label': 'caption',
     'comp_fileid': 'img0001-alto',
     'comp_page_no': 1,
     'coords': [820, 5899, 1244, 40]}]},
  'c': [298, 4676, 2241, 1159],
  'pOf': 'ILOL-1843-05-21-a-i0001'},
 {'m': {'id': 'ILOL-1843-05-21-a-i0003',
   'tp': 'image',
   'pp': [1],
   'iiif_link': 'https://impresso-pro

In [ ]:
ci_39 = [i for i in bl_issue.issue_data['i'] if i['m']['id']=='ILOL-1843-05-21-a-i0039']
ci_39

In [10]:
for p in bl_issue.pages:
    p.add_issue(bl_issue)
    p.parse()

#bl_issue.pages[0].page_data

Block P1_CB00001 is of an image: Illustration, ignoring its textlines
Block P1_CB00002 is of an image: Illustration, ignoring its textlines
Block P1_CB00003 is of an image: Illustration, ignoring its textlines
Block P1_CB00004 is of an image: Illustration, ignoring its textlines
Block P1_CB00005 is of an image: Illustration, ignoring its textlines
Block P1_CB00006 is of an image: Illustration, ignoring its textlines
Block P1_TB00007 not in mappings: {'pa0001001': 'ILOL-1843-05-21-a-i0001', 'pa0001002': 'ILOL-1843-05-21-a-i0001', 'pa0001003': 'ILOL-1843-05-21-a-i0001', 'pa0001004': 'ILOL-1843-05-21-a-i0001', 'pa0001005': 'ILOL-1843-05-21-a-i0002', 'pa0001006': 'ILOL-1843-05-21-a-i0002', 'pa0001007': 'ILOL-1843-05-21-a-i0003', 'pa0001008': 'ILOL-1843-05-21-a-i0004', 'pa0001009': 'ILOL-1843-05-21-a-i0004', 'pa0001010': 'ILOL-1843-05-21-a-i0004', 'pa0001011': 'ILOL-1843-05-21-a-i0004', 'pa0001012': 'ILOL-1843-05-21-a-i0004', 'pa0001013': 'ILOL-1843-05-21-a-i0005', 'pa0002001': 'ILOL-1843-0

In [14]:
bl_issue.pages[0].page_data['r']

[{'c': [1775, 2258, 526, 40],
  'p': [{'c': [1775, 2258, 526, 40],
    'l': ({'c': [1775, 2258, 526, 40],
      't': [{'c': [1775, 2258, 216, 39], 'tx': 'LONDON,'},
       {'c': [2014, 2261, 107, 33], 'tx': 'MAY'},
       {'c': [2140, 2265, 52, 33], 'tx': '21,'},
       {'c': [2214, 2261, 87, 33], 'tx': '1843.'}]},)}]},
 {'c': [3257, 2265, 326, 38],
  'p': [{'c': [3257, 2265, 326, 38],
    'l': ({'c': [3257, 2265, 326, 38],
      't': [{'c': [3257, 2265, 123, 38], 'tx': '[PRICE'},
       {'c': [3403, 2265, 180, 31], 'tx': 'POURPENC'}]},)}]},
 {'c': [2636, 2356, 1167, 438],
  'p': [{'c': [2636, 2356, 1167, 438],
    'l': ({'c': [2636, 2356, 1167, 46],
      't': [{'c': [2636, 2362, 124, 31], 'tx': 'mental'},
       {'c': [2781, 2365, 112, 33], 'tx': 'water,'},
       {'c': [2907, 2362, 66, 30], 'tx': 'and'},
       {'c': [2987, 2373, 49, 20], 'tx': 'we'},
       {'c': [3049, 2364, 115, 30], 'tx': 'record'},
       {'c': [3178, 2373, 36, 20], 'tx': 'as'},
       {'c': [3228, 2373, 20, 20

In [11]:
pg_1_ci_regions = {}

for region in bl_issue.pages[0].page_data['r']:
    if 'pOf' in region:
        og_ci_id = region['pOf']
    else:
        og_ci_id = "No attached CI"
    if og_ci_id in pg_1_ci_regions:
        pg_1_ci_regions[og_ci_id].append(region)
    else:
        pg_1_ci_regions[og_ci_id] = [region]

pg_1_ci_regions.keys()

dict_keys(['No attached CI', 'ILOL-1843-05-21-a-i0001', 'ILOL-1843-05-21-a-i0002', 'ILOL-1843-05-21-a-i0004', 'ILOL-1843-05-21-a-i0005'])

In [56]:
def find_regions_per_ci(page_regions):
    ci_regions = {}
    unattached_counter = 1
    for region in page_regions:

        if 'pOf' in region:
            og_ci_id = region['pOf']
        else:
            og_ci_id = f"No attached CI {unattached_counter}"
            unattached_counter += 1

        if og_ci_id in ci_regions:
            ci_regions[og_ci_id].append(region)
        else:
            ci_regions[og_ci_id] = [region]
    return ci_regions

##### Printing the content items and their boxes on the pages

In [49]:
colors = ["red", 'green', 'blue', 'purple', 'orange', 'cyan', 'brown', "limegreen", 'pink']

In [57]:
def draw_box_on_img(
    base_img_path: str, coords_xy: list, img: Image = None, width: int = 10, color="red", text=None
) -> Image:
    """Draw a bounding box on an image given coordinates in x1y1x2y2 format.

    The image can either be provided through its path, or as a PIL.Image object (specifically
    if other bboxes have already been drawn on it.)

    Args:
        base_img_path (str): Path to the image to open as a PIL Image.
        coords_xy (list): Coordinates of the bbox to draw.
        img (Image, optional): PIL image if already loaded. Defaults to None.
        width (int, optional): Stroke width for the bbox. Defaults to 10.
        color (str, optional): Color of stroke. Defaults to 'red'.

    Returns:
        Image: Resulting PIL Image with the bbox drawn on it.
    """
    if not img:
        img = Image.open(base_img_path)
        img = img.convert('RGB')
        img = img.convert(mode='1')
        img = img.convert('RGB')

    img_draw = ImageDraw.Draw(img)
    img_draw.rectangle(coords_xy, outline=color, width=width)

    if text is not None:
        #font=ImageFont.truetype("arial.ttf",10 ) 
        img_draw.text((coords_xy[0]+50, coords_xy[1]+75), text, 'black')#,font=font)
    return img

In [54]:
ci_regions = find_regions_per_ci(bl_issue.pages[0].page_data['r'])

[{'c': [1775, 2258, 526, 40], 'p': [{'c': [1775, 2258, 526, 40], 'l': ({'c': [1775, 2258, 526, 40], 't': [{'c': [1775, 2258, 216, 39], 'tx': 'LONDON,'}, {'c': [2014, 2261, 107, 33], 'tx': 'MAY'}, {'c': [2140, 2265, 52, 33], 'tx': '21,'}, {'c': [2214, 2261, 87, 33], 'tx': '1843.'}]},)}]}, {'c': [3257, 2265, 326, 38], 'p': [{'c': [3257, 2265, 326, 38], 'l': ({'c': [3257, 2265, 326, 38], 't': [{'c': [3257, 2265, 123, 38], 'tx': '[PRICE'}, {'c': [3403, 2265, 180, 31], 'tx': 'POURPENC'}]},)}]}, {'c': [2636, 2356, 1167, 438], 'p': [{'c': [2636, 2356, 1167, 438], 'l': ({'c': [2636, 2356, 1167, 46], 't': [{'c': [2636, 2362, 124, 31], 'tx': 'mental'}, {'c': [2781, 2365, 112, 33], 'tx': 'water,'}, {'c': [2907, 2362, 66, 30], 'tx': 'and'}, {'c': [2987, 2373, 49, 20], 'tx': 'we'}, {'c': [3049, 2364, 115, 30], 'tx': 'record'}, {'c': [3178, 2373, 36, 20], 'tx': 'as'}, {'c': [3228, 2373, 20, 20], 'tx': 'a'}, {'c': [3263, 2364, 77, 36], 'tx': 'fact,'}, {'c': [3359, 2361, 115, 41], 'tx': 'highly'}, {'c

In [ ]:
for page_object in bl_issue.pages:
    print(f"Page: {page_object.id}")

    ci_regions = find_regions_per_ci(page_object.page_data['r'])
    
    page_info = test_issue_renaming_info[str(page_object.number)]
    img_dir_path = os.path.join(page_info['img_dir_path'], page_info['new_filename'])
    
    img = Image.open(img_dir_path)
    img = img.convert('RGB')
    #img = img.convert(mode='1')
    #img = img.convert('RGB')
    for ci_num, (ci, ci_regions) in enumerate(ci_regions.items()):
        """if len(ci_regions) == 1:
            regions = ci_regions[0]
        else:
            regions = ci_regions
            print(ci, len(regions))"""

        for r_num, region in enumerate(ci_regions):
            
            region_coords = coords_to_xy(region['c'])
            #img = draw_box_on_img(img_dir_path, region_coords, img, text=f"{ci}-reg{r_num}")
            reg_color = colors[ci_num%len(colors)]
            print(f"{ci}-reg{r_num} --> {reg_color}")
            if reg_color == 'orange':
                print(region['c'])
            img = draw_box_on_img(img_dir_path, region_coords, img, color=reg_color, text=f"{ci}-reg{r_num}")
            #t_coords =  (region_coords[0], region_coords[1] + 100)
            #img.text(t_coords, f"{ci}-reg{r_num}", fill=color_id, font_size=100)

    print(f"page {page_object.id} regions and CIs")
    img.show()
    break

In [15]:
test_issue_renaming_info

{'12': {'original_filename': '0003005_18430521_0012.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0012.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '1': {'original_filename': '0003005_18430521_0001.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0001.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843/05/21',
  'width': 4409,
  'height': 6504},
 '5': {'original_filename': '0003005_18430521_0005.jp2',
  'new_filename': 'ILOL-1843-05-21-a-p0005.jp2',
  'issue_id': 'ILOL-1843-05-21-a',
  'original_nlp': '0003005',
  'img_dir_path': '/mnt/impresso_images_BL/ILOL/1843/05/21/a',
  'ocr_dir_path': '/mnt/project_impresso/original/BL/ILOL/0003005/1843

### TEST image coords for BL-alias format

In [ ]:
img_path = "/mnt/impresso_ocr_BL_old/0000104/1881/0729/0000104_18810729_0001.jp2"

page_coords = coords_to_xy([1525,347,7808,8125])
summer_coords = [94,96,383,150]
the_coords = [800,90,929,130]
eagle_coords = [945,92,1136,135]
packing_coords = [1149,96,1377,141]
sentence_coords = [the_coords[0], the_coords[1], packing_coords[2], packing_coords[3]]
article_1_coords = [52,62,6335,7829]

img = draw_box_on_img(img_path, article_1_coords, width=20)

img

In [ ]:
img_pat_2 = "/mnt/impresso_ocr_BL_old/0000104/1881/0729/0000104_18810729_0002.jp2"

article_2_coords = [0,45,4945,7872]
article_3_coords = [4183,35,5642,7822]
article_4_coords = [4903,25,6350,7817]

img_2 = draw_box_on_img(img_pat_2, article_3_coords, width=20)
img_2 = draw_box_on_img(img_pat_2, article_4_coords, img_2, width=10)

img_2